# 03 · Stage 3 — fine-grained classifiers (one per group)

Each model is trained on the images that truly belong to its group
(ground-truth routing) and predicts only that group's diseases.

**Source-balanced negatives (G5).** Media haze (MH) comes almost entirely from
RFMiD and cataract (CAT) from ODIR, so a positives-only G5 model reaches a
meaningless validation AUC of 1.000 by recognising the camera. Adding negatives
sampled *per source* equalises the positive rate within each dataset and removes
the shortcut; this de-confounded model is the one used in the cascade.

In [ ]:
import sys, json
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent / "src"))   # or: pip install -e ..

import numpy as np, pandas as pd, torch
from fundus import config, labels
from fundus.data import (load_split, unhealthy_only, FundusDataset, make_loader,
                         train_transform, eval_transform, pos_weight)
from fundus.engine import set_seed, device, fit, predict

P = config.Paths()          # reads $FUNDUS_DATA_ROOT and $FUNDUS_RUN_ROOT
set_seed(config.SEED)
DEV = device()
print("device:", DEV, "| data:", P.data_root.resolve())

In [ ]:
from fundus.models import build_classifier, load_weights
from fundus.losses import make_criterion
from fundus.engine import evaluate_multilabel
from fundus.data import source_balanced_negatives
from fundus.metrics import best_f1_threshold

cfg = config.STAGE3
tr_all = unhealthy_only(load_split(P.csv("train")))
va_all = unhealthy_only(load_split(P.csv("val")))
LOW_SUPPORT = 15            # <= this many training positives -> keep threshold at 0.5

# (group key, train with source-balanced negatives?). G5 is trained both ways:
# the positives-only run exists only to demonstrate the source shortcut;
# the cascade uses G5_neg.
PLAN = [("G1", False), ("G2", False), ("G3", False), ("G4", False),
        ("G5", False), ("G5", True), ("G6", False)]

In [ ]:
def group_frames(key, with_negatives):
    cls = labels.GROUPS[labels.GKEY[key]]
    if with_negatives:
        return cls, source_balanced_negatives(tr_all, cls), source_balanced_negatives(va_all, cls)
    pick = lambda d: d[d[cls].sum(1) > 0].reset_index(drop=True)
    return cls, pick(tr_all), pick(va_all)


def train_and_calibrate(key, with_negatives):
    cls, tr, va = group_frames(key, with_negatives)
    tds = FundusDataset(tr, P.images("train"), train_transform(cfg.img_size), label_cols=cls)
    vds = FundusDataset(va, P.images("val"),   eval_transform(cfg.img_size),  label_cols=cls)
    tdl = make_loader(tds, cfg.batch_size, True,  cfg.num_workers)
    vdl = make_loader(vds, cfg.batch_size, False, cfg.num_workers)
    print(f"\n[{key}] classes={cls} train={len(tds)} val={len(vds)} negatives={with_negatives}")
    print(tr.groupby("source")[cls].sum().to_string())

    out = P.run(f"stage3/{key}{'_neg' if with_negatives else ''}")
    model = build_classifier(len(cls), cfg)
    crit = make_criterion("bce", pos_weight(tds.labels, cfg.pos_weight_clip))
    fit(model, tdl, vdl, crit, evaluate_multilabel, out, cfg, monitors=("macro_auc", "macro_ap"))

    # checkpoint and thresholds come from the SAME model (best validation macro-AP)
    model = load_weights(model, out / "best_macro_ap.pt", DEV)
    m, (Y, Pv, S) = evaluate_multilabel(model, vdl, DEV)
    n_train = tds.labels.sum(0)
    thr = {c: (best_f1_threshold(Y[:, j], Pv[:, j]) if n_train[j] > LOW_SUPPORT and Y[:, j].sum() > 0 else 0.5)
           for j, c in enumerate(cls)}
    json.dump({"thresholds": thr, "val": m}, open(out / "thresholds.json", "w"), indent=2)
    print({k: round(v, 4) for k, v in m.items()})
    return out

RUNS = [train_and_calibrate(k, neg) for k, neg in PLAN]

## Shortcut probe for G5

Feed the positives-only and the de-confounded G5 models images that contain
*no* G5 disease. A model that learned the dataset will call non-cataract ODIR
images "CAT" and non-haze RFMiD images "MH"; within-source AUC cannot be produced
by recognising the source.

In [ ]:
cls = labels.GROUPS["G5_Media"]
probe = va_all[va_all[cls].sum(1) == 0]
for tag in ["G5", "G5_neg"]:
    d = P.run_root / "stage3" / tag
    if not (d / "best_macro_ap.pt").exists():
        continue
    import dataclasses
    model = load_weights(build_classifier(len(cls), dataclasses.replace(cfg, pretrained=False)),
                         d / "best_macro_ap.pt", DEV)
    for src in labels.SOURCES:
        sub = probe[probe.source == src]
        if sub.empty:
            continue
        ds = FundusDataset(sub, P.images("val"), eval_transform(cfg.img_size), label_cols=cls)
        _, pr, _ = predict(model, make_loader(ds, 8, False, cfg.num_workers), DEV)
        print(f"{tag:7s} {src:7s} n={len(ds):4d}  called MH {100*(pr[:,0]>=.5).mean():5.1f}%  "
              f"called CAT {100*(pr[:,1]>=.5).mean():5.1f}%")